*This notebook is the lesson [8. Tuples and destructuring](https://simontreanor.github.io/Pyfun/learn/08-tuples.html) from Learn Pyfun, for the Pyfun kernel. Run the cells from the top: definitions carry forward from cell to cell, and a cell with a hole `?` shows what belongs there instead of running.*

# 8. Tuples and destructuring

A tuple groups a fixed number of values that can have different types, written `(a, b)`, exactly the
Python tuple. A record gives its fields names, while a tuple identifies them by position, which
suits a short pairing like a name and a score. Because the parentheses are also grouping, Pyfun
reads them by how many elements are inside: `()` is unit, the empty value from lesson 1, `(x)` is
just `x` in parentheses, and a real tuple has two or more elements. There is no one element tuple.

You take a tuple apart the same way you take an ADT apart, by matching. A single variable pattern
covers a tuple completely, so one `case` is exhaustive:

In [ ]:
let swap p =
  match p:
    case (a, b): (b, a)

let names = ["ada", "alan"]
let scores = [10, 9]

let line pair =
  match pair:
    case (name, score): f"{name}: {score}"

let lines = List.zip names scores |> List.map line

let byName = List.zip names scores |> Map.ofList
let adaScore = Option.withDefault 0 (Map.tryFind "ada" byName)

print (swap (1, 2))
print lines
byName |> Map.toList |> print
print adaScore

```console
(2, 1)
['ada: 10', 'alan: 9']
[('ada', 10), ('alan', 9)]
10
```

`List.zip` pairs two lists element by element into a list of tuples, so `List.zip names scores` is
`[("ada", 10), ("alan", 9)]`. Mapping `line` over that list destructures each pair in the `case`
and binds `name` and `score`, the way Python's `for name, score in pairs` unpacks as it goes.

Tuples also bridge lists and maps. `Map.ofList` builds a `Map` from a list of pairs, and
`Map.toList` turns a map back into its pairs, so a zip followed by `Map.ofList` is a compact way to
build a lookup table from two parallel lists.

## Binding both parts at once

When a pattern covers every value, as a tuple pattern does, a whole `match` is more ceremony than
the job needs. A `let` takes the same pattern directly, which is how Python writes it too:

In [ ]:
type Point = { x: int, y: int }

let (name, score) = ("ada", 10)
let (first, (second, third)) = (1, (2, 3))
let Point { x, y } = Point { x = 3, y = 4 }

let swap p =
  let (a, b) = p
  (b, a)

print f"{name} scored {score}"
print (first + second + third)
print (x * y)
print (swap (1, 2))

```console
ada scored 10
6
12
(2, 1)
```

A function parameter takes the same patterns, which is where they save the most typing. A parameter
that destructures binds the parts on the way in, so the body starts with the pieces already named
and the `match` from the previous example disappears:

In [ ]:
type Cell = { letter: string, points: int }

let line (name, score) = f"{name}: {score}"
let label (Cell { letter, points }) = f"{letter} is worth {points}"

print (line ("ada", 10))
print (label (Cell { letter = "Q", points = 10 }))
print (List.map line [("ada", 10), ("alan", 9)])

```console
ada: 10
Q is worth 10
['ada: 10', 'alan: 9']
```

That last line is the everyday payoff: a function folding or mapping over pairs reads as a function
of two named things, with no helper in the way. Note the brackets around a record pattern in a
parameter, which keep `label (Cell { … })` from parsing as two parameters.

The rule in all three positions is that the pattern has to match every value of its type. Names,
`_`, tuples and records qualify, and they nest. A constructor pattern does not, because it can
fail:

In [ ]:
let (Some x) = Some 1

```console
error: a `let` binding must always match, so it takes a name, `_`, or a tuple or record of those,
not a constructor pattern (use `match` instead, which has somewhere to fall through to)
```

Leaving the brackets off does not help. A name after `let` that starts with a capital letter can
only be a constructor, so `let Some x = …` is refused too, with a message that says so.

That is the trade. `match` handles the patterns that can fail, because it has other arms to fall
through to, and `let` handles the ones that cannot, because it does not.

## Lists destructure too

The same `match` works over a `List`. A list pattern names elements in brackets, just like the
literal that builds one: `[]` is the empty list, `[only]` matches a list of exactly one, and a
`*rest` star soaks up any number of elements. Python allows one star per pattern, and so does Pyfun,
but it can sit anywhere: `[x, *rest]` is head and tail, `[*init, last]` is everything but the last,
and `[first, *mid, last]` binds the two ends with the middle collected between them.

In [ ]:
let describe xs =
  match xs:
    case []: "empty"
    case [only]: f"just {only}"
    case [first, *mid, last]: f"{first} ... {last}"

print (describe [])
print (describe [7])
print (describe [1, 2, 3, 4])

```console
empty
just 7
1 ... 4
```

An empty-list arm plus a star arm covers every list, so a match built from `[]` and one starred
pattern is exhaustive with no `case _`. The `[first, *mid, last]` arm needs at least two elements,
which is why the `[]` and `[only]` arms come first to catch the shorter lists.

## Exercise

`List.zip` has paired each name with its score, and the `case (name, score)` arm has already
destructured a pair for you. Fill the hole so each pair becomes a line like `ada: 10`, then the
program joins the lines with a comma. Use the bound `name` and `score` in an interpolated string.

In [ ]:
let names = ["ada", "alan"]
let scores = [10, 9]

let line pair =
  match pair:
    case (name, score): ?

let lines = List.zip names scores |> List.map line

lines |> String.join ", " |> print

The checker reports:

```console
note: hole `?` has type `'a`
 --> 6:25
  |
6 |     case (name, score): ?
  |                         ^
1 unfilled hole
```

Expected output:

```console
ada: 10, alan: 9
```

[Open in the playground](https://simontreanor.github.io/Pyfun/playground/#code=bGV0IG5hbWVzID0gWyJhZGEiLCAiYWxhbiJdCmxldCBzY29yZXMgPSBbMTAsIDldCgpsZXQgbGluZSBwYWlyID0KICBtYXRjaCBwYWlyOgogICAgY2FzZSAobmFtZSwgc2NvcmUpOiA_CgpsZXQgbGluZXMgPSBMaXN0LnppcCBuYW1lcyBzY29yZXMgfD4gTGlzdC5tYXAgbGluZQoKbGluZXMgfD4gU3RyaW5nLmpvaW4gIiwgIiB8PiBwcmludAo)

<details>
<summary>Show solution</summary>

```pyfun
let names = ["ada", "alan"]
let scores = [10, 9]

let line pair =
  match pair:
    case (name, score): f"{name}: {score}"

let lines = List.zip names scores |> List.map line

lines |> String.join ", " |> print
```

The `case (name, score)` arm binds both parts of the pair, and the f-string builds one line per
pair before `String.join` stitches them together.
</details>